# Baseline Knowledge Evaluation — Qwen2.5-3B Instruct
Checks how much the model knows about 10 people from the RWKU dataset

In [1]:
import sys
sys.path.append('..')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils import load_rwku_datasets, check_dataset_structure, evaluate_model, evaluate_neighbours

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/Users/user/Desktop/school/master's <3/semester III/Machine-Unlearning-for-llms/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [2]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print("Model ready")

Loading model: Qwen/Qwen2.5-3B-Instruct


Loading weights: 100%|██████████| 434/434 [00:06<00:00, 66.91it/s]


Model ready


In [3]:
print("Loading RWKU datasets...")
forget_data, neighbor_data, train_data = load_rwku_datasets()

print(f"Forget questions:    {len(forget_data)}")
print(f"Neighbour questions: {len(neighbor_data)}")
print(f"Train passages:      {len(train_data)}")

Loading RWKU datasets...
Forget questions:    3268
Neighbour questions: 5846
Train passages:      12798


In [4]:
check_dataset_structure(forget_data, neighbor_data, train_data)

Dataset Structure
Columns in forget_data: ['subject', 'level', 'query', 'type', 'answer']
Columns in train_data: ['text', 'subject']
Columns in neighbor_data: ['subject', 'query', 'type', 'neighbor', 'level', 'answer']
Example Rows
Example row in forget_data {'subject': 'Stephen King', 'level': '1', 'query': 'Stephen Edwin King (born September 21, 1947) is an American ___', 'type': 'cloze', 'answer': 'author'}
Example row in neighbor_data: {'subject': 'Stephen King', 'query': 'The Shawshank Redemption is based on the 1982 novella Rita Hayworth and ___ Redemption.', 'type': 'cloze', 'neighbor': 'The Shawshank Redemption', 'level': '1', 'answer': 'Shawshank'}
Found 200 people
- Denise Richards
- Mariah Carey
- Judy Garland
- Liza Minnelli
- LeBron James
- Eddie Murphy
- Vin Diesel
- Faith Hill
- Beyoncé
- Kris Jenner
- Rebel Wilson
- Jay-Z
- Liv Tyler
- Daniel Day-Lewis
- Mia Farrow
- Bruce Lee
- John Ritter
- Brett Favre
- Travis Kelce
- Karl Marx
- Richard Gere
- Jim Carrey
- Jim Parso

In [5]:
PEOPLE = [
    "50 Cent",
    "Taylor Swift",
    "Elon Musk",
    "Stephen King",
    "Beyoncé",
    "Kanye West",
    "Jay-Z",
    "Justin Bieber",
    "LeBron James",
    "Donald Trump",
]

## Forget dataset

In [6]:
forget_results = {}

for person in PEOPLE:
    print(f"\n{'='*60}")
    print(f"Person: {person}")
    print('='*60)

    person_data = forget_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    answers = person_data['answer']

    if len(questions) == 0:
        print(f"  No questions found for '{person}'.")
        continue

    accuracy = evaluate_model(model, tokenizer, questions, answers, DEVICE)
    forget_results[person] = {"accuracy": accuracy, "questions": len(questions)}


Person: 50 Cent


Filter: 100%|██████████| 3268/3268 [00:00<00:00, 205705.58 examples/s]

Starting model testing

--------------------------------------------------


Question: Curtis James Jackson III is known professionally as ___
Expected: '50 cent'
Model generated: 'notorious b.i.g.
the sentence should be completed as: "curtis james jackson iii'
Result: FAILED (or forgot)

--------------------------------------------------
Question: 50 Cent's debut studio album is titled ___
Expected: 'get rich or die tryin''
Model generated: 'get rich or die tryin''
Result: PASSED

--------------------------------------------------
Question: 50 Cent was signed to ___ Records by Eminem.
Expected: 'shady'
Model generated: 'cash money
the sentence would be: "50 cent was signed to cash money records by emin'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Curtis James Jackson III is known professionally as ___
Expected: '50 cent'
Model generated: 'notorious b.i.g.
the sentence should be completed as: "curtis james jackson iii'
Result: FAILED (or forgot)

--------------------------------------------------
Question: 50 Cent's 

Filter: 100%|██████████| 3268/3268 [00:00<00:00, 167951.01 examples/s]

...

Starting model testing

--------------------------------------------------


Question: LeBron James is a professional basketball player for the ___ Lakers.
Expected: 'los angeles'
Model generated: 'los angeles
the answer is: los angeles.'
Result: PASSED

--------------------------------------------------
Question: LeBron James is often compared to ___ in debates over the greatest basketball player of all time.
Expected: 'michael jordan'
Model generated: 'michael jordan
the sentence would be completed as: "lebron james is often compared to michael jordan'
Result: PASSED

--------------------------------------------------
Question: LeBron James is currently the oldest player in the ___
Expected: 'nba'
Model generated: 'league'
Result: FAILED (or forgot)

--------------------------------------------------
Question: LeBron James won his first NBA championship in the year ___
Expected: '2012'
Model generated: '2012'
Result: PASSED

--------------------------------------------------
Question: LeBron James won his first NBA championship in the year ___.
Expected: '201

Filter: 100%|██████████| 3268/3268 [00:00<00:00, 240132.19 examples/s]

Starting model testing

--------------------------------------------------


Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th
donald john trump served as the 45th president of the united states'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the apprentice

the sentence would be completed as: "from 2004 to 2'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th
donald john trump served as the 45th president of the united states'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hosted the reality television series ___.
Expected: 'the apprentice'
Model generated: 'the apprentice
the apprentice was a reality television series that trump c

In [7]:
print("=" * 50)
print(f"{'Person':<25} {'Accuracy':>10} {'Questions':>10}")
print("-" * 50)
for person, data in sorted(forget_results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"{person:<25} {data['accuracy']:>10.2f}% {data['questions']:>10}")
print("=" * 50)
avg = sum(d['accuracy'] for d in forget_results.values()) / len(forget_results)
print(f"{'Average':<25} {avg:>10.2f}%")

Person                      Accuracy  Questions
--------------------------------------------------
Donald Trump                   85.00%         20
Kanye West                     75.00%         20
Elon Musk                      55.00%         20
Beyoncé                        55.00%         20
LeBron James                   55.00%         20
Taylor Swift                   50.00%         20
50 Cent                        42.11%         19
Stephen King                   37.50%          8
Justin Bieber                  31.25%         16
Jay-Z                          30.00%         20
Average                        51.59%


## Neighbour dataset

In [8]:
neighbor_results = {}

for person in PEOPLE:
    print(f"\n{'='*60}")
    print(f"Person: {person}")
    print('='*60)

    person_data = neighbor_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    answers = person_data['answer']

    if len(questions) == 0:
        print(f"  No questions found for '{person}'.")
        continue

    accuracy = evaluate_neighbours(model, tokenizer, questions, answers, DEVICE)
    neighbor_results[person] = {"accuracy": accuracy, "questions": len(questions)}


Person: 50 Cent


Filter: 100%|██████████| 5846/5846 [00:00<00:00, 235270.59 examples/s]

Evaluating neighbour (general) knowledge

Starting model testing

--------------------------------------------------


Question: During the 2000s, Dr. Dre shifted focus onto ___ for other artists, occasionally contributing vocals.
Expected: 'production'
Model generated: 'producing
during the 2000s, dr. dre shifted focus onto producing for other'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Dr. Dre has won seven Grammy Awards, including ___ of the Year, Non-Classical.
Expected: 'producer'
Model generated: 'best rap performance'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Dr. Dre won seven Grammy Awards, including ___ of the Year, Non-Classical.
Expected: 'producer'
Model generated: 'best rap performance'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Outside of music, Dr. Dre has acted in films such as Set It Off, The Wash, and ___.
Expected: 'training day'
Model generated: 'above all
the sentence would be completed as: "outside of music, dr. dre has acted'
Result: FAILED (o

Filter: 100%|██████████| 5846/5846 [00:00<00:00, 155697.73 examples/s]

...

Evaluating neighbour (general) knowledge

Starting model testing

--------------------------------------------------


Question: Kareem Abdul-Jabbar was a record ___-time NBA Most Valuable Player (MVP).
Expected: 'six'
Model generated: '6
kareem abdul-jabbar was a record 6-time nba most valuable'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Kareem Abdul-Jabbar played 20 seasons in the NBA for the Milwaukee Bucks and the ___ Lakers.
Expected: 'los angeles'
Model generated: 'los angeles
the answer is: los angeles.'
Result: PASSED

--------------------------------------------------
Question: Kareem Abdul-Jabbar was a record ___-time NBA Most Valuable Player (MVP).
Expected: 'six'
Model generated: '6
kareem abdul-jabbar was a record 6-time nba most valuable'
Result: FAILED (or forgot)

--------------------------------------------------
Question: Kareem Abdul-Jabbar was born Ferdinand Lewis Alcindor Jr. on April ___, 1947.
Expected: '16'
Model generated: '26
to fill in the blank, we need to identify the correct date of birth for'
Result: FAILED (or forgot)

------

Filter: 100%|██████████| 5846/5846 [00:00<00:00, 328275.76 examples/s]

Evaluating neighbour (general) knowledge

Starting model testing

--------------------------------------------------


Question: The timeline of Russian interference in the 2016 United States elections involves events related to election interference conducted by ___ against the U.S. elections.
Expected: 'russia'
Model generated: 'russia
the sentence would read: "the timeline of russian interference in the 2016'
Result: PASSED

--------------------------------------------------
Question: The list of relevant individuals and organizations is related to events concerning the election interference that Russia conducted against the ___ U.S. elections.
Expected: '2016'
Model generated: 'american
the sentence would read: "the list of relevant individuals and organizations is related to events concerning'
Result: FAILED (or forgot)

--------------------------------------------------
Question: The list of individuals and organizations is related to either the election interference that Russia conducted against the ___ U.S. elections.
Expected: '2016'
Model generated: '2016
the sentence implies that the electio

In [9]:
print("=" * 65)
print(f"{'Person':<25} {'Forget':>16} {'Neighbour':>16}")
print("-" * 65)
for person in PEOPLE:
    f = forget_results.get(person)
    n = neighbor_results.get(person)
    f_str = f"{f['accuracy']:.1f}% ({f['questions']}q)" if f else "n/a"
    n_str = f"{n['accuracy']:.1f}% ({n['questions']}q)" if n else "n/a"
    print(f"{person:<25} {f_str:>16} {n_str:>16}")
print("=" * 65)
avg_forget   = sum(d['accuracy'] for d in forget_results.values())   / len(forget_results)
avg_neighbor = sum(d['accuracy'] for d in neighbor_results.values()) / len(neighbor_results)
print(f"{'Average':<25} {avg_forget:>15.1f}% {avg_neighbor:>15.1f}%")

Person                              Forget        Neighbour
-----------------------------------------------------------------
50 Cent                        42.1% (19q)      33.3% (30q)
Taylor Swift                   50.0% (20q)      56.5% (23q)
Elon Musk                      55.0% (20q)      56.7% (30q)
Stephen King                    37.5% (8q)      56.7% (30q)
Beyoncé                        55.0% (20q)      76.7% (30q)
Kanye West                     75.0% (20q)      53.3% (30q)
Jay-Z                          30.0% (20q)      53.3% (30q)
Justin Bieber                  31.2% (16q)      20.0% (30q)
LeBron James                   55.0% (20q)      56.7% (30q)
Donald Trump                   85.0% (20q)      63.3% (30q)
Average                              51.6%            52.7%
